In [46]:
# import libraries
import pandas as pd
import numpy as np
import plotly.express as px

In [47]:
# prepare data

madrid_2018 = pd.read_csv('madrid_2018.csv')
stations = pd.read_csv('stations.csv')
# pollutants used for European Air Quality Index(EAQI)
pollutants = ['O_3', 'NO_2', 'SO_2', 'PM25_rolling_avg', 'PM10_rolling_avg']
# breakpoints for each pollutant based on EAQI standards
breakpoints = {
    'O_3':   [50, 100, 130, 240, 380],
    'NO_2':  [40, 90, 120, 230, 340],
    'SO_2':  [100, 200, 350, 500, 750],
    'PM25_rolling_avg': [20, 40, 50, 100, 150],
    'PM10_rolling_avg': [10, 20, 25, 50, 75],
}

In [48]:

madrid_2018.head()

,date,BEN,CH4,CO,EBE,NMHC,NO,NO_2,NOx,O_3,PM10,PM25,SO_2,TCH,TOL,station
0,2018-03-01 01:00:00,NaN,NaN,0.3,NaN,NaN,1.0,29.0,31.0,NaN,NaN,NaN,2.0,NaN,NaN,28079004
1,2018-03-01 01:00:00,0.5,1.39,0.3,0.2,0.02,6.0,40.0,49.0,52.0,5.0,4.0,3.0,1.41,0.8,28079008
2,2018-03-01 01:00:00,0.4,NaN,NaN,0.2,NaN,4.0,41.0,47.0,NaN,NaN,NaN,NaN,NaN,1.1,28079011
3,2018-03-01 01:00:00,NaN,NaN,0.3,NaN,NaN,1.0,35.0,37.0,54.0,NaN,NaN,NaN,NaN,NaN,28079016
4,2018-03-01 01:00:00,NaN,NaN,NaN,NaN,NaN,1.0,27.0,29.0,49.0,NaN,NaN,3.0,NaN,NaN,28079017


In [49]:
stations.head()

,id,name,address,lon,lat,elevation
0,28079004,Pza. de España,Plaza de España,-3.712247,40.423853,635
1,28079008,Escuelas Aguirre,Entre C/ Alcalá y C/ O’ Donell,-3.682319,40.421564,670
2,28079011,Avda. Ramón y Cajal,Avda. Ramón y Cajal esq. C/ Príncipe de Vergara,-3.677356,40.451475,708
3,28079016,Arturo Soria,C/ Arturo Soria esq. C/ Vizconde de los Asilos,-3.639233,40.440047,693
4,28079017,Villaverde,C/. Juan Peñalver,-3.713322,40.347139,604


In [50]:
# format data
madrid_2018['date'] = pd.to_datetime(madrid_2018['date'])

In [51]:
# calculate 24 hour rolling average for PM10 and PM2.5
madrid_2018['PM10_rolling_avg'] = madrid_2018.groupby('station')['PM10'].transform(lambda x: x.rolling(window=24, min_periods=1).mean())
madrid_2018['PM25_rolling_avg'] = madrid_2018.groupby('station')['PM10'].transform(lambda x: x.rolling(window=24, min_periods=1).mean())

In [52]:
# function to calculate EAQI for a given pollutant value
def get_pollutant_score(value, pollutant):
        if pd.isna(value): return np.nan
        limits = breakpoints[pollutant]
        for i, limit in enumerate(limits):
            if value <= limit:
                return i + 1
        return 6


def calculate_eaqi(df):
    station_means = df.groupby('station')[pollutants].mean()

    def score_station(row):
        scores = [get_pollutant_score(row[p], p) for p in pollutants]
        valid  = [s for s in scores if not pd.isna(s)]
        return max(valid) if valid else np.nan

    station_means['EAQI_Score'] = station_means.apply(score_station, axis=1)
    return station_means

In [53]:
eaqi_score = calculate_eaqi(madrid_2018)
eaqi_score.head()

,O_3,NO_2,SO_2,PM25_rolling_avg,PM10_rolling_avg,EAQI_Score
station,,,,,,
28079004,NaN,40.464534,2.536161,NaN,NaN,2
28079008,38.288555,57.308362,4.298774,11.027672,11.027672,2
28079011,NaN,43.347720,NaN,NaN,NaN,2
28079016,43.111537,40.276078,NaN,NaN,NaN,2
28079017,40.409617,40.008014,4.432404,NaN,NaN,2


In [54]:
# merge EAQI scores with station metadata
merged = eaqi_score.merge(stations, left_index=True, right_on='id')
merged.head()

,O_3,NO_2,SO_2,PM25_rolling_avg,PM10_rolling_avg,EAQI_Score,id,name,address,lon,lat,elevation
0,NaN,40.464534,2.536161,NaN,NaN,2,28079004,Pza. de España,Plaza de España,-3.712247,40.423853,635
1,38.288555,57.308362,4.298774,11.027672,11.027672,2,28079008,Escuelas Aguirre,Entre C/ Alcalá y C/ O’ Donell,-3.682319,40.421564,670
2,NaN,43.347720,NaN,NaN,NaN,2,28079011,Avda. Ramón y Cajal,Avda. Ramón y Cajal esq. C/ Príncipe de Vergara,-3.677356,40.451475,708
3,43.111537,40.276078,NaN,NaN,NaN,2,28079016,Arturo Soria,C/ Arturo Soria esq. C/ Vizconde de los Asilos,-3.639233,40.440047,693
4,40.409617,40.008014,4.432404,NaN,NaN,2,28079017,Villaverde,C/. Juan Peñalver,-3.713322,40.347139,604


In [63]:
# visualize EAQI scores by station location
eaqi_colors = [
    "#00CC66", # 1 - Very Good
    "#FFFF66", # 2 - Good
    "#FF9933", # 3 - Moderate
    "#FF3333", # 4 - Poor
    "#990000", # 5 - Very Poor
    "#7E0023"  # 6 - Extremely Poor
]

fig = px.scatter_map(
    merged,
    lat="lat",
    lon="lon",
    color="EAQI_Score",
    hover_name="name",
    hover_data={
        "lat": False,
        "lon": False,
        "EAQI_Score": True
    },
    color_continuous_scale=eaqi_colors,
    range_color=[1, 6],
    zoom=11,
    center={"lat": 40.435, "lon": -3.69},
    map_style="carto-positron",
    title="Madrid Air Quality: EAQI Score & NO2 Levels"
)
fig.update_traces(marker={'size': 35})
fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig.show()